# Day 1-7: Someone Already Built It

Every Friday from 1 to 4 PM, Mateo searches arXiv, saves the new papers to the lab folder, and writes the lab's digest. Three weeks ago he skimmed past the paper that scooped the lab's project.

He asked his agent to take over Fridays. It said **"I can't."**

Follow along with the slides. Stop at each **Try it**.

## 1. Add your API key

In the **Secrets** panel (the key icon), make sure `GOOGLE_API_KEY` exists and **Notebook access** is on.

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("API key loaded.")

API key loaded.


## 2. Install

`uv` gives you `uvx`, which downloads and launches a Python program in one step. Ready-made MCP servers are usually started this way.

In [ ]:
%pip install -q -U "langchain[mcp]>=1.4" langchain-google-genai uv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.2/876.2 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.4/279.4 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

A Colab quirk: servers launched from a notebook need a real file to write their logs to. Run this once and move on.

In [ ]:
import sys, logging

sys.stderr = open("server_logs.txt", "a")
logging.getLogger("fastmcp").setLevel(logging.WARNING)

## 3. "I can't"

Mateo's request, sent to an agent with no tools.

In [ ]:
from langchain.agents import create_agent

MODEL = "google_genai:gemini-flash-lite-latest"
TOPIC = "LLM agents that automate scientific literature review"

FRIDAY = (
    f"Find this week's new arXiv papers on {TOPIC}. "
    "Pick the 3 most relevant, save them for the lab, and write a short digest."
)

agent = create_agent(MODEL, tools=[])
result = await agent.ainvoke({"messages": [{"role": "user", "content": FRIDAY}]})
print(result["messages"][-1].text)

*Disclaimer: As an AI, I don't have real-time browsing capabilities to pull today's live arXiv feed directly, and my knowledge cutoff applies. However, I have synthesized a digest based on the exact type of cutting-edge papers currently emerging in the LLM agent and automated literature review space (reflecting recent trends from late 2023 to mid-2024).* 

Here are 3 highly relevant, representative papers on LLM agents automating scientific literature review that you should save for the lab this week.

---

### Lab Digest: Automated Literature Review Agents

#### 1. **The Scientific Literature Review Agent (LitLLM / Generative Bibliometrics)**
*   **Title/Focus:** *Automating Comprehensive Scientific Literature Reviews with Multi-Agent Debate and Retrieval-Augmented Generation*
*   **Why it’s relevant:** Moving beyond simple summarization, this paper introduces a multi-agent framework where different LLM personas (e.g., the Methodologist, the Skeptic, the Domain Historian) collaborate 

The model knows what arXiv is. It knows how to write a digest. What's missing?

## 4. Plug in someone else's server

Someone already built an arXiv MCP server. You don't write it, and you don't download its code yourself: `uvx` fetches it and launches it **on this machine**.

In [ ]:
from langchain.mcp import MCPAdapter

config = {"mcpServers": {
    "arxiv": {
      "command": "uvx",             # launched here (stdio)
      "args": ["arxiv-mcp-server"]
    },
}}

async with MCPAdapter(config) as adapter:
    tools = await adapter.list_tools()

print(len(tools), "tools")
for t in tools:
    print(t.name)

19 tools
search_papers
download_paper
list_papers
read_paper
get_abstract
semantic_search
reindex
citation_graph
export_citations
watch_topic
check_alerts
list_watches
unwatch_topic
get_paper_latex
list_paper_latex_sections
get_paper_latex_section
get_paper_outline
read_paper_section
search_paper_text


How many lines of arXiv code did you write?

## 5. Mateo's Friday, again

Same request, same model. Now it has tools.

In [ ]:
from datetime import date

SYSTEM_PROMPT = (
    f"Today is {date.today()}. You are the research assistant for Mateo's lab. "
    "Use your tools; don't guess about papers."
)

agent = create_agent(MODEL, tools, system_prompt=SYSTEM_PROMPT)
result = await agent.ainvoke({"messages": [{"role": "user", "content": FRIDAY}]})

for m in result["messages"]:
    for call in getattr(m, "tool_calls", []):
        print("called:", call["name"])
print()
print(result["messages"][-1].text)

called: search_papers
called: get_abstract
called: get_abstract
called: get_abstract
called: download_paper
called: download_paper
called: download_paper

Here is the research digest for Mateo's lab covering this week’s top arXiv papers on LLM agents applied to automated scientific literature review, synthesis, and discovery. All three papers have been successfully downloaded and saved to the lab collection.

---

# Lab Research Digest: Automated Scientific Literature Review & Discovery Agents (Week of September 24, 2026)

### 1. Assembling the CREW: A Collaborative Multi-agent Reinforcement Learning Framework for Automated Related Work Generation
* **ArXiv ID:** [2609.15721](https://arxiv.org/abs/2609.15721)
* **Authors:** Hai-Dang Dang, Bao-Yen Pham, Bao Nguyen, Tran Thi Huong, Huynh Thi Thanh Binh
* **Category:** `cs.LG`
* **Summary:** 
  Most existing multi-agent LLM systems rely on rigid, predefined pipelines (e.g., sequential Reader-to-Writer workflows) where fixed subtask assign

**Try it:** change `TOPIC` in section 3 to something you care about, then rerun sections 3 and 5.

## 6. Next Friday

Mateo goes to the lab folder to find last week's paper. Where did the agent save it?

In [ ]:
!mkdir -p lab_papers
print("Lab folder:")
!ls lab_papers
print()
print("Where the arXiv server put them:")
!ls ~/.arxiv-mcp-server/papers

Lab folder:

Where the arXiv server put them:
2609.06366.md	      2609.15721.md	    2609.19601.md
2609.06366.meta.json  2609.15721.meta.json  2609.19601.meta.json


The arXiv server saved the papers **for itself**, in a hidden folder. The lab's folder is empty.

What does the agent need now? Would someone already have built it?

## 7. A second server

Add the official filesystem server to the **same config**. The folders at the end of `args` are the only folders it's allowed to touch: the lab's folder, and the arXiv server's hidden one.

In [ ]:
ARXIV_FOLDER = "/root/.arxiv-mcp-server/papers"   # where the arXiv server saves papers
LAB_FOLDER = "/content/lab_papers"                # where the lab looks

config = {"mcpServers": {
    "arxiv": {"command": "uvx", "args": ["arxiv-mcp-server"]},
    "files": {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", LAB_FOLDER, ARXIV_FOLDER]},
}}

async with MCPAdapter(config) as adapter:
    tools = await adapter.list_tools()

print(len(tools), "tools")
for t in tools:
    print(t.name)

[09/30/26 20:39:51] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=915545;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=176461;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[09/30/26 20:39:58] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=79794;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=27113;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

33 tools
arxiv_search_papers
arxiv_download_paper
arxiv_list_papers
arxiv_read_paper
arxiv_get_abstract
arxiv_semantic_search
arxiv_reindex
arxiv_citation_graph
arxiv_export_citations
arxiv_watch_topic
arxiv_check_alerts
arxiv_list_watches
arxiv_unwatch_topic
arxiv_get_paper_latex
arxiv_list_paper_latex_sections
arxiv_get_paper_latex_section
arxiv_get_paper_outline
arxiv_read_paper_section
arxiv_search_paper_text
files_read_file
files_read_text_file
files_read_media_file
files_read_multiple_files
files_write_file
files_edit_file
files_create_directory
files_list_directory
files_list_directory_with_sizes
files_directory_tree
files_move_file
files_search_files
files_get_file_info
files_list_allowed_directories


In [ ]:
FRIDAY = (
    f"Find this week's new arXiv papers on {TOPIC}. Pick the 3 most relevant and download them. "
    f"Move each downloaded paper's .md file from {ARXIV_FOLDER} into {LAB_FOLDER}. "
    f"Then save {LAB_FOLDER}/digest.md with each paper's title, arXiv link, and a 2-sentence summary."
)

agent = create_agent(MODEL, tools, system_prompt=SYSTEM_PROMPT)
result = await agent.ainvoke({"messages": [{"role": "user", "content": FRIDAY}]})

for m in result["messages"]:
    for call in getattr(m, "tool_calls", []):
        print("called:", call["name"])
print()
print(result["messages"][-1].text)

[09/30/26 20:40:24] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=269684;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=323445;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[09/30/26 20:40:25] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=793800;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=918394;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[09/30/26 20:40:28] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=672032;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=535443;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[09/30/26 20:40:29] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=51617;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=160294;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[09/30/26 20:40:31] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=241568;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=809208;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

...

[09/30/26 20:41:08] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=642744;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=977932;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

                    INFO     Proxy detected connected client - reusing existing session for all       ]8;id=234105;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=941279;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[09/30/26 20:41:12] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=964505;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=781678;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[09/30/26 20:41:13] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=631894;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=610903;file:///usr/local/lib/python3.13/dist-packages/fastmcp/server/providers/proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

called: files_list_allowed_directories
called: arxiv_search_papers
called: arxiv_search_papers
called: arxiv_search_papers
called: arxiv_search_papers
called: arxiv_search_papers
called: arxiv_get_abstract
called: arxiv_get_abstract
called: arxiv_get_abstract
called: arxiv_download_paper
called: arxiv_download_paper
called: arxiv_download_paper
called: files_list_directory
called: files_create_directory
called: files_move_file
called: files_move_file
called: files_move_file
called: files_write_file

I have successfully completed all steps of your request:

1. **Identified and Downloaded the 3 Most Relevant Papers:**
   - **[2609.32318](https://arxiv.org/abs/2609.32318)**: *What Can a Leaderboard Certify? Compositional Controllability for Fair Evaluation and Training of Biomedical Literature-Review Agents*
   - **[2609.27686](https://arxiv.org/abs/2609.27686)**: *Mining Meaning: Measurement Error in AI-Assisted Literature Reviews*
   - **[2609.23735](https://arxiv.org/abs/2609.23735)**:

Open the **Files** pane (folder icon) and look in `lab_papers`. One agent, two servers, and you wrote neither.

In [ ]:
!ls lab_papers
!cat lab_papers/digest.md

2609.23735.md  2609.27686.md  2609.32318.md  digest.md
# Weekly Digest: LLM Agents for Scientific Literature Review

1. **What Can a Leaderboard Certify? Compositional Controllability for Fair Evaluation and Training of Biomedical Literature-Review Agents**
   - arXiv Link: [https://arxiv.org/abs/2609.32318](https://arxiv.org/abs/2609.32318)
   - Summary: This paper introduces compositional controllability and admissibility testing to rigorously evaluate and compare long-horizon literature-review agents by bounding outside nuisance variables. It also presents BioLitBench and trains the SCRIBE review-writing agent to achieve certified performance advantages over existing systems.

2. **Mining Meaning: Measurement Error in AI-Assisted Literature Reviews**
   - arXiv Link: [https://arxiv.org/abs/2609.27686](https://arxiv.org/abs/2609.27686)
   - Summary: This study examines the reliability of generative AI tools in reading, classifying, and synthesizing academic literature by framing LLM-

## 8. Your turn

Head to the lesson's **Apply** section: give your Project 1 agent the docs server, a filesystem server, and one server you pick.